# Presence Test

This notebook connects a second peer ("notebook-agent") back to itself via `runtimed`,
then animates cursor presence across cells. You should see a colored cursor bar and
gutter dots appear as the agent peer moves around.

**Requirements:** Open this notebook in the nteract dev app. The daemon will resolve
`runtimed` from `python/runtimed/pyproject.toml` automatically.

In [1]:
import runtimed
import time
import os

In [2]:
# Connect a second peer to this same notebook.
# The daemon client auto-detects the socket path.
client = runtimed.DaemonClient()
rooms = client.list_rooms()
print(f"Open notebooks: {len(rooms)}")
for r in rooms:
    print(f"  {r['notebook_id']}")

Open notebooks: 2
  /Users/kylekelley/code/src/github.com/nteract/worktrees/desktop/new-branch-who
-dis/notebooks/whatevs.ipynb
  /Users/kylekelley/code/src/github.com/nteract/worktrees/desktop/new-branch-who
-dis/python/runtimed/demos/test-presence.ipynb


In [6]:
# Pick the first room (this notebook) and connect as a second peer
notebook_id = None

for room in rooms:
    if "test-presence" in room["notebook_id"]:
        notebook_id = room["notebook_id"]

if notebook_id is None:
    raise ValueError("Test Presence not found")
session = runtimed.Session(notebook_id=notebook_id, peer_label="\U0001f916 Agent")
session.connect()
print(f"Connected to {notebook_id} as second peer")

Connected to /Users/kylekelley/code/src/github.com/nteract/worktrees/desktop/new
-branch-who-dis/python/runtimed/demos/test-presence.ipynb as second peer


In [8]:
# List cells — you should see a gutter dot appear briefly on each cell
import time

cells = session.get_cells()
print(f"{len(cells)} cells:")
for i, c in enumerate(cells):
    time.sleep(0.5)
    preview = c.source[:50].replace("\n", "\u21b5") if c.source else "(empty)"
    print(f"  [{i}] {c.cell_type}: {preview}")

9 cells:
  [0] markdown: # Presence Test↵↵This notebook connects a second p
  [1] code: import runtimed↵import time↵import os
  [2] code: # Connect a second peer to this same notebook.↵# T
  [3] code: # Pick the first room (this notebook) and connect
  [4] code: # List cells — you should see a gutter dot appear
  [5] code: # Animate cursor across cells — watch the gutter d
  [6] code: # Animate cursor within this cell — watch the curs
  [7] code: # Selection demo — grow a highlight across a cell↵
  [8] code: # Clean up — disconnect the second peer↵session.cl


In [10]:
# Animate cursor across cells — watch the gutter dots jump
print("Jumping between cells...")
for c in cells:
    session.set_cursor(c.id, line=0, column=0)
    time.sleep(0.8)
print("Done — did you see the dots?")

Jumping between cells...
Done — did you see the dots?


In [12]:
# Animate cursor within this cell — watch the cursor bar sweep
code_cells = [c for c in cells if c.cell_type == "code" and len(c.source) > 20]
if code_cells:
    target = code_cells[0]
    lines = target.source.split("\n")
    print(f"Sweeping cursor across '{lines[0][:30]}...' ({len(lines)} lines)")
    for line_num, line_text in enumerate(lines[:5]):
        for col in range(0, len(line_text) + 1, 2):
            session.set_cursor(target.id, line=line_num, column=col)
            time.sleep(0.03)
    print("Sweep complete")
else:
    print("No code cells with enough content to demo")

Sweeping cursor across 'import runtimed...' (3 lines)
Sweep complete


In [13]:
# Selection demo — grow a highlight across a cell
if code_cells:
    target = code_cells[0]
    lines = target.source.split("\n")
    print("Growing selection...")
    for line_num, line_text in enumerate(lines[:4]):
        for col in range(0, len(line_text) + 1, 3):
            session.set_selection(
                target.id,
                anchor_line=0,
                anchor_col=0,
                head_line=line_num,
                head_col=col,
            )
            time.sleep(0.05)
    time.sleep(1)
    # Clear by setting cursor
    session.set_cursor(target.id, line=0, column=0)
    print("Selection cleared")

Growing selection...
Selection cleared


In [ ]:
# Clean up — disconnect the second peer
session.close()
print("Second peer disconnected — cursor should disappear")